# **word2vec nlp tutorial**

> ????

???????50,000?IMDB??????,??????????????????,????IMDB??<5???????0,?????7??????1.???????????30????25,000???????????????25,000??????????????,????50,000?IMDB?????????????

> ????

| ?? | ?? |
| :--- | :--- |
| labeledTrainData | ?????????????????,????????,????25,000?,?????????ID,?????? |
| testData | ??????????????????,????25,000?,????????????????????????????? |
| unlabeledTrainData | ???????????????????????????,??50,000?,????????????  |
| sampleSubmission | ?????????????????? |

> ????

| ?? | ?? |
| :--- | :--- |
| id | ???????ID |
| sentiment | ?????; 1?????,0????? |
| review | ????? |


# ??????

????:??????????
????: K??(knn)?????(LogisticRegression)?????(RandomForest)??????(SVM)?xgboost?GBDT

> ??:

```
?. ????
1. ???????
2. ????:?????????,??????
3. ??????,????????:????????????,????????????????

?. ????
1.????,??,????????????????.
2.???????????????(??????,???????)

?. ????
1.????????????,????????????,????????????.
2.?????????,????????????????.

?. ????

?. ?????????
1.????????????,???????.
2.?????????,???????
```

* * * 

* ????: https://www.kaggle.com/c/word2vec-nlp-tutorial
* ????: https://www.cnblogs.com/zhao441354231/p/6056914.html
* ????: https://blog.csdn.net/lijingpengchina/article/details/52250765


## ?.????

### ???????

* ???????: https://www.kaggle.com/c/word2vec-nlp-tutorial/data


In [ ]:
# ???????
import pandas as pd
import numpy as np
from bs4 import *

### ????

In [ ]:
import os
data_dir = "/opt/data/kaggle/getting-started/word2vec-nlp-tutorial"
# ?????       
train = pd.read_csv(os.path.join(data_dir, 'labeledTrainData.tsv'), header=0, delimiter="\t", quoting=3)
pre = pd.read_csv(os.path.join(data_dir, 'testData.tsv'), header=0, delimiter="\t", quoting=3)

In [ ]:
print(train.shape, '\t', pre.shape, '\t')
print(train.info(), '\n')
print('\n', train.columns.values)
print('\n', train.head(3))

### ?????

* 1.??html??
* 2.????
* 3.????/token
* 4.?????
* 5.???????

In [ ]:
# def review_to_wordlist(review):
#     '''
#     ?IMDB????????
#     ??:http://blog.csdn.net/longxinchen_ml/article/details/50629613
#     '''
#     # ??HTML??,????
#     review_text = BeautifulSoup(review, "html.parser").get_text()
#     # ???????????????
#     review_text = re.sub("[^a-zA-Z]", " ", review_text)
#     # ???????,????list
#     words = review_text.lower().split()
#     # ??words
#     return words


# ?????
label = train['sentiment']
train_data = []
pre_data = []
for i in range(len(train['review'])):
    train_data.append(BeautifulSoup(train['review'][i], "html.parser").get_text())
test_data = []
for i in range(len(pre['review'])):
    pre_data.append(BeautifulSoup(pre['review'][i], "html.parser").get_text())

In [ ]:
# ????
print(train_data[0], '\n')
print(pre_data[0])

## ????

In [ ]:
# ????????????TFIDF?????
data_all = train_data + pre_data
len_train = len(train_data)

????????????,?????????,??????????????,??????????????,??: 

1. Bow-of-Words?? 
2. TF-IDF?? 
3. Word2vec?? 

In [ ]:
from nltk.corpus import stopwords
#?????,set()?????????
list_stopWords = list(set(stopwords.words('english')))
print(list_stopWords)

In [ ]:
from gensim import corpora

# bow ??    
import re
texts = [[word for word in re.sub("[^a-zA-Z]", " ", doc.lower()) if word != "" and word not in list_stopWords] for doc in data_all]
dictionary = corpora.Dictionary(texts)
# ???????,????????
corpus = [dictionary.doc2bow(text) for text in texts]

In [ ]:
for key in dictionary.keys()[0:5]:
    print (key, dictionary[key])

print(corpus[0])

### 2.TF-IDF + Lsi????

In [ ]:
from gensim import models
tfidf_model = models.TfidfModel(corpus=corpus, id2word=dictionary, normalize=True) 
# ???corpus??tf-idf??
corpus_tfidf = tfidf_model[corpus]

In [ ]:
## lsi ????,??????
lsi_model = models.LsiModel(corpus_tfidf, id2word=dictionary, num_topics=200)
corpus_lsi = lsi_model[corpus_tfidf]

# ????,???numpy???
all_x = [[v for k,v in doc] for doc in corpus_lsi]

In [ ]:
# print(np.shape(corpus_lsi))
# (50000, 200, 2)
print(lsi_model.print_topics(3))
# print(corpus_lsi[0])

import numpy as np
print(np.shape(all_x))

### 3.Word2vec??

???????? L = SUM[log(p(w|contect(w))],??w??????????w???,???????,????????p(w|contect(w), Word2vec???????????????

In [ ]:
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

# def show_diff(origin, html, text):
#     print(origin)
#     print("\n-----------show diff-----------\n")
#     print(html)
#     print("\n-----------show diff-----------\n")
#     print(text)

# origin = train['review'][0]
# html = BeautifulSoup(origin, "html.parser").get_text()
# text = re.sub('[^a-zA-Z]', ' ', html).strip()
# show_diff(origin, html, text)

stopwords = set(stopwords.words("english"))

def review_to_sentence(review):
    html = BeautifulSoup(review, "html.parser").get_text()
    text = re.sub('[^a-zA-Z]', ' ', html).strip()
    words = [word for word in text.lower().split() if word not in stopwords]
    return words

unlabeled_train = pd.read_csv(os.path.join(data_dir, 'unlabeledTrainData.tsv'), header=0, delimiter="\t", quoting=3 )
train_texts = pd.concat([train['review'], unlabeled_train['review']])
sentences = list(map(review_to_sentence, train_texts))

In [ ]:
print(np.shape(sentences))

In [ ]:
import time
from gensim.models import Word2Vec
# ????
num_features = 784    # Word vector dimensionality(?????300?,????CNN, ?? 784? = 28*28)               
min_word_count = 10   # Minimum word count                        
num_workers = 4       # Number of threads to run in parallel
context = 10          # Context window size                                                                                    
downsampling = 1e-3   # Downsample setting for frequent words

In [ ]:
%%time
# ????
print("?????...")
# model = Word2Vec(sentences, workers=num_workers, \
#             size=num_features, min_count=min_word_count, \
#             window=5, sample=downsampling)
model = Word2Vec(sentences, size=num_features, window=5)
print("????")

In [ ]:
np.shape(model["film"])

In [ ]:
model.wv.doesnt_match("man woman child kitchen".split())

In [ ]:
model.wv.doesnt_match("france england germany berlin".split())

In [ ]:
model.wv.doesnt_match("paris berlin london austria".split())

In [ ]:
model.wv.most_similar("man", topn=5)

In [ ]:
model.wv.most_similar("queen", topn=5)

In [ ]:
model.wv.most_similar("awful", topn=5)

In [ ]:
model.wv.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)

In [ ]:
def makeFeatureVec(words, model, num_features):
    '''
    ?????????????????
    '''
    featureVec = np.zeros((num_features,), dtype="float32")
    nwords = 0.

    # Index2word??????????,??????,???set?
    index2word_set = set(model.wv.index2word)
    for word in words:
        if word in index2word_set:
            nwords = nwords + 1.
            featureVec = np.add(featureVec, model[word])

    # ???
    featureVec = np.divide(featureVec, nwords)
    return featureVec


def getAvgFeatureVecs(reviews, model, num_features):
    '''
    ????????,????????????,????????????????
    '''
    counter = 0
    reviewFeatureVecs = np.zeros((len(reviews), num_features), dtype="float32")

    for review in reviews:
        if counter % 5000 == 0:
            print("Review %d of %d" % (counter, len(reviews)))

        reviewFeatureVecs[counter] = makeFeatureVec(review, model, num_features)
        counter = counter + 1

    return reviewFeatureVecs

In [ ]:
%time 
trainDataVecs = getAvgFeatureVecs(texts[:len_train], model, num_features)
print(np.shape(trainDataVecs))

In [ ]:
%time 
testDataVecs = getAvgFeatureVecs(texts[len_train:], model, num_features)
print(np.shape(testDataVecs))

In [ ]:
from sklearn.naive_bayes import GaussianNB as GNB
from sklearn.cross_validation import cross_val_score

gnb_model = GNB()
gnb_model.fit(trainDataVecs, label)

scores = cross_val_score(gnb_model, trainDataVecs, label, cv=10, scoring='roc_auc')
print("\n???????? 10???????: \n", scores)
print("\n???????? 10?????????: \n", np.mean(scores))

## ???? - ????

### KNN ????

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(all_x[:len_train], label)

scores = cross_val_score(knn_model, all_x[:len_train], label, cv=10, scoring='roc_auc')
print("\nknn?? 10???????: \n", scores)
print("\nknn?? 10?????????: \n", np.mean(scores))

### ??? ????

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

tree_model = DecisionTreeClassifier()
tree_model.fit(all_x[:len_train], label)

scores = cross_val_score(tree_model, all_x[:len_train], label, cv=10, scoring='roc_auc')
print("\n??? 10???????: \n", scores)
print("\n??? 10?????????: \n", np.mean(scores))

### ???? ????

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr_model = LogisticRegression(C=0.1, max_iter=100)
lr_model.fit(all_x[:len_train], label)

scores = cross_val_score(lr_model, all_x[:len_train], label, cv=10, scoring='roc_auc')
print("\n???? 10???????: \n", scores)
print("\n???? 10?????????: \n", np.mean(scores))


# from sklearn.model_selection import GridSearchCV

# # ??grid search???
# grid_values = {'C': [1, 15, 30, 50]}  
# """
# penalty: l1 or l2, ?????????????
# """
# model_LR = GridSearchCV(estimator=LR(penalty='l2', dual=True, random_state=0), grid_values, scoring='roc_auc', cv=20)
# model_LR.fit(train_x, label)

# ????
# print(model_LR.cv_results_, '\n', model_LR.best_params_, model_LR.best_score_)

### SVM ????

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# model = SVC(C=4, kernel='rbf')
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(all_x[:len_train], label)

scores = cross_val_score(svm_model, all_x[:len_train], label, cv=10, scoring='roc_auc')
print("\nSVM 10???????: \n", scores)
print("\nSVM 10?????????: \n", np.mean(scores))

In [ ]:
svm_model = SVC(kernel='linear', probability=True)

### XGBoost ????

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
import numpy as np

xgb_model = XGBClassifier(n_estimators=150, min_samples_leaf=3, max_depth=6)
"""
AttributeError: 'list' object has no attribute 'shape'
list => np.array
"""
xgb_model.fit(np.array(all_x[:len_train]), label)

scores = cross_val_score(xgb_model, np.array(all_x[:len_train]), label, cv=10, scoring='roc_auc')
print("\nXGB 10???????: \n", scores)
print("\nXGB 10?????????: \n", np.mean(scores))

## ????

### bagging: ???? 

????????,????????

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# parameters= {'n_estimators': range(10, 101, 10)}  
# gsearch_rf = GridSearchCV(
#     estimator=RandomForestClassifier(max_depth=8, random_state=0),
#     param_grid=parameters, scoring='roc_auc', cv=10)

# gsearch_rf = gsearch_rf.fit(all_x[:len_train], label)

# print(gsearch_rf.cv_results_, '\n', gsearch_rf.best_params_, '\t', gsearch_rf.best_score_)
"""
[mean: 0.87486, std: 0.00576, params: {'n_estimators': 10}, mean: 0.88505, std: 0.00611, params: {'n_estimators': 20}, mean: 0.89032, std: 0.00609, params: {'n_estimators': 30}, mean: 0.89246, std: 0.00537, params: {'n_estimators': 40}, mean: 0.89439, std: 0.00528, params: {'n_estimators': 50}, 
 mean: 0.89507, std: 0.00607, params: {'n_estimators': 60}, mean: 0.89591, std: 0.00618, params: {'n_estimators': 70}, mean: 0.89634, std: 0.00634, params: {'n_estimators': 80}, mean: 0.89671, std: 0.00607, params: {'n_estimators': 90}, mean: 0.89753, std: 0.00588, params: {'n_estimators': 100}] 

{'n_estimators': 100}  0.89753344
"""

rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=0)
rf_model.fit(all_x[:len_train], label)

scores = cross_val_score(svm_model, all_x[:len_train], label, cv=10, scoring='roc_auc')
print("\n???? 10???????: \n", scores)
print("\n???? 10?????????: \n", np.mean(scores))

### boosting: AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

ab_model = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=2),
    n_estimators=600,
    learning_rate=1)

ab_model.fit(all_x[:len_train], label)

scores = cross_val_score(ab_model, all_x[:len_train], label, cv=10, scoring='roc_auc')
print("\nAdaBoost 10???????: \n", scores)
print("\nAdaBoost 10?????????: \n", np.mean(scores))

### voting: ?????

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import VotingClassifier

"""
soft????????????????????????????????
????????????????,????????
??svc??,????????,??????,??????????????
"""

vot_model = VotingClassifier(
#     estimators=[('lr', lr_model), ('svm', svm_model), ('xgb', xgb_model), ('rf', rf_model), ('ab', ab_model)]
    estimators=[('xgb', xgb_model), ('rf', rf_model)],
    voting='hard')
vot_model.fit(np.array(all_x[:len_train]), np.array(label))

scores = cross_val_score(vot_model, np.array(all_x[:len_train]), np.array(label), cv=10, scoring='roc_auc')
print("\nAdaBoost 10???????: \n", scores)
print("\nAdaBoost 10?????????: \n", np.mean(scores))

### stacking: ??

In [ ]:
y = np.array([0, 0, 1, 1])
skf = StratifiedKFold(y, 2)
len(skf)

In [ ]:
'''??????????????'''
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.cross_validation import StratifiedKFold

# ??train???,????,?????????????
X = np.array(all_x[:len_train])
X_predict = np.array(all_x[len_train:])
label.astype(np.integer)
y = label.values

# clfs = [LogisticRegression(C=0.1, max_iter=100),
#         xgb.XGBClassifier(max_depth=6, n_estimators=100, num_round = 5),
#         RandomForestClassifier(n_estimators=100, max_depth=6, oob_score=True),
#         GradientBoostingClassifier(learning_rate=0.3, max_depth=6, n_estimators=100)]

clfs = [knn_model, tree_model, lr_model, svm_model, xgb_model, rf_model, ab_model]


# ??n_folds
n_folds = 10
skf = StratifiedKFold(y, n_folds)


# ?????
dataset_blend_train = np.zeros((X.shape[0], len(clfs)))
dataset_blend_test = np.zeros((X_predict.shape[0], len(clfs)))

# ????
for j, clf in enumerate(clfs):
    '''?????????'''
    # print(j, clf)
    dataset_blend_test_j = np.zeros((X_predict.shape[0], len(skf)))
    for i, (train, test) in enumerate(skf):
        '''???i???????,??????????,???????????i???????'''
        # print("Fold", i)
        X_train, y_train, X_test, y_test = X[train], y[train], X[test], y[test]
        clf.fit(X_train, y_train)
        y_submission = clf.predict_proba(X_test)[:, 1]
        dataset_blend_train[test, j] = y_submission
        dataset_blend_test_j[:, i] = clf.predict_proba(X_predict)[:, 1]
    '''?????,????k????????????????'''
    dataset_blend_test[:, j] = dataset_blend_test_j.mean(1)

# ????????
stacking_model = LogisticRegression(C=0.1, max_iter=100)
stacking_model.fit(dataset_blend_train, y_train)

scores = cross_val_score(ab_model, dataset_blend_train, label, cv=10, scoring='roc_auc')
print("\nAdaBoost 10???????: \n", scores)
print("\nAdaBoost 10?????????: \n", np.mean(scores))

## ????

In [ ]:
test_predicted = np.array(model_NB.predict(corpus_tfidf[len_train:]))
print('????...')

import os
root_dir = "/opt/data/kaggle/getting-started/word2vec-nlp-tutorial"
        
submission_df = pd.DataFrame(data ={'id': test['id'], 'sentiment': test_predicted})
print(submission_df.head(10))
submission_df.to_csv(os.path.join(root_dir, 'submission_br.csv'), index = False)

print('??.')

## CNN ??? ????: https://zhuanlan.zhihu.com/p/26729228

### ??,??????????

In [ ]:
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

stopwords = set(stopwords.words("english"))

def review_to_sentence(review):
    html = BeautifulSoup(review, "html.parser").get_text()
    text = re.sub('[^a-zA-Z]', ' ', html).strip()
    words = [word for word in text.lower().split() if word not in stopwords]
    return words

unlabeled_train = pd.read_csv(os.path.join(data_dir, 'unlabeledTrainData.tsv'), header=0, delimiter="\t", quoting=3 )
train_texts = pd.concat([train['review'], unlabeled_train['review']], axis=0, ignore_index=True)
sentences = list(map(review_to_sentence, train_texts))

In [ ]:
np.shape(sentences), len(sentences[0]), len(sentences[1]), len(train['review']), len( unlabeled_train['review'])

### ???????

In [ ]:
from gensim import corpora
dictionary = corpora.Dictionary(sentences)

In [ ]:
dictionary.add_documents([[" "]])

In [ ]:
for key in dictionary.keys()[0:5]:
    print (key, dictionary[key])


In [ ]:
len(dictionary.token2id), len(dictionary.id2token), type(dictionary.token2id), type(dictionary.id2token)

In [ ]:
dictionary.token2id["another"], dictionary.id2token[3], dictionary.token2id[" "]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F     # ???????
from torch.autograd import Variable
import torch.utils.data as Data
import torchvision     # ?????

In [ ]:
max_len = max([len(i) for i in sentences])
max_index = dictionary.token2id[" "]
max_list = [max_index for x in range(max_len)]
print(max_len, "\n", len(max_list))

In [ ]:
# # prepare_sequence ?????????? Variable ??
# def prepare_sequence(seq):
#     idxs = [dictionary.token2id[w] for w in seq]
#     if len(idxs) < max_len:
#         idxs = idxs + max_list[len(idxs):]
# #     print('?????????:', idxs)
#     tensor = torch.LongTensor(idxs)
#     return Variable(tensor)

# sentence_in = prepare_sequence(sentences[1383])
# # word_embeddings = nn.Embedding(len(dictionary.token2id), 5)
# # word_embeddings(sentence_in)
# x = cnn(sentence_in)
# print(x)

In [ ]:
# sentence_in.data.size(0)

In [ ]:
# sentences[0][:3], "\n\n", len(sentence_in)

### ?????

In [ ]:
# prepare_sequence ?????????? Variable ??
def prepare_sequence(seq):
    idxs = [dictionary.token2id[w] for w in seq]
    if len(idxs) < max_len:
        idxs = idxs + max_list[len(idxs):]
#     print('?????????:', idxs)
    return idxs


from sklearn.model_selection import train_test_split

X_train = list(map(prepare_sequence, sentences[:len(train)]))
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_train, label.tolist(), test_size=0.2, shuffle=True, random_state=42)


In [ ]:
type(X_train_d), type(y_train_d)

In [ ]:
np.shape(X_train_d), np.shape(X_test_d), np.shape(y_train_d), np.shape(y_test_d)

In [ ]:
X_train_d[0][:3], X_test_d[0][:3], y_train_d[:3], y_test_d[:3]

In [ ]:
class textCNN(nn.Module):
    
    def __init__(self, vocab_size, embedding_dim, max_len, n_classes):
        super(textCNN, self).__init__()
        
        self.model_name = 'alexnet'
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.max_len = max_len
        
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        self.features = nn.Sequential(
#             nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(kernel_size=3, stride=2),

#             nn.Conv2d(64, 192, kernel_size=5, padding=2),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(kernel_size=3, stride=2),

#             nn.Conv2d(192, 384, kernel_size=3, padding=1),
#             nn.ReLU(inplace=True),

#             nn.Conv2d(384, 256, kernel_size=3, padding=1),
#             nn.ReLU(inplace=True),

#             nn.Conv2d(256, 256, kernel_size=3, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(kernel_size=3, stride=2),
            
            
            nn.Conv2d(1, 16, 5, 1, 2),
            nn.ReLU(),    # activation
            nn.MaxPool2d(kernel_size=2, stride=2),    # ? 2x2 ???????, output shape (16, 14, 14), ?????2
    
            nn.Conv2d(16, 32, 5, 1, 2),  # output shape (32, 14, 14)
            nn.ReLU(),  # activation
            nn.MaxPool2d(2),  # output shape (32, 7, 7)

            nn.Conv2d(32, 64, 5, 1, 2),  
            nn.ReLU(),  
            nn.MaxPool2d(2), 
    
            nn.Conv2d(64, 128, 5, 1, 2),  
            nn.ReLU(),  
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
#             nn.Dropout(),
#             nn.Linear(256 * 6 * 6, 4096),
#             nn.ReLU(inplace=True),

#             nn.Dropout(),
#             nn.Linear(4096, 4096),
#             nn.ReLU(inplace=True),

#             nn.Linear(4096, n_classes),
            
            nn.Dropout(),
            nn.Linear(45056, n_classes),   
        )
 
    def forward(self, x):
        x = self.word_embeddings(x)
        x = x.view(len(x), 1, self.max_len, self.embedding_dim)
        x = self.features(x)
        x = x.view(x.size(0), -1)
        output = self.classifier(x)
        return output     # return x for visualization


In [ ]:
cnn = textCNN(len(dictionary.token2id), 64, max_len, 2)
print(cnn)  # net architecture

# optimizer = torch.optim.SGD(cnn.parameters(), lr=0.02)  # ?? net ?????, ???
# lr ????
# weight_decay(????): ?? L2 regularization  (1e-5?? 1*(10?-5??)?0.00001)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-5, weight_decay=1e-7)
# ??????, ?????!??! one-hot ???, ??1D Tensor, (batch,)
# ??????2D tensor (batch, n_classes)
loss_func = nn.CrossEntropyLoss()

In [ ]:
import time
import math

def timeSince(since):
    now = time.time()
    s = now - since
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

start = time.time()

In [ ]:
tr_x = torch.LongTensor(X_train_d)
tr_y = torch.LongTensor(y_train_d)
te_x = torch.LongTensor(X_test_d)
te_y = torch.LongTensor(y_test_d)

torch_train_dataset = Data.TensorDataset(data_tensor=tr_x, target_tensor=tr_y)
torch_test_dataset = Data.TensorDataset(data_tensor=te_x, target_tensor=te_y)

BATCH_SIZE = 20      # ????????

# ? dataset ?? DataLoader
train_loader = Data.DataLoader(
    dataset=torch_train_dataset,      # torch TensorDataset format
    batch_size=BATCH_SIZE,            # ?? batch ???????
    shuffle=True,                     # ??????? (?????)
    num_workers=2,                    # ???????
)
test_loader = Data.DataLoader(
    dataset=torch_test_dataset,       # torch TensorDataset format
    batch_size=BATCH_SIZE,            # ?? batch ???????
    shuffle=True,                     # ??????? (?????)
    num_workers=2,                    # ???????
)

In [ ]:
from sklearn.metrics import log_loss

Epoch = 5
print_every = 20
max_step = len(X_train_d)/BATCH_SIZE

# ???????
current_loss = 0
all_losses = []

for epoch in range(Epoch):
    for step, (batch_x,  batch_y) in enumerate(train_loader):  # ??? loader ???????????
        b_x = Variable(batch_x)   # batch x
        b_y = Variable(batch_y)   # batch y
        
        out = cnn(b_x)                 # ?? net ???? x, ?????
        loss = loss_func(out, b_y)     # ???????

        optimizer.zero_grad()   # ?????????????
        loss.backward()         # ??????, ???????
        optimizer.step()        # ????????? net ? parameters ?

        current_loss += loss.data[0]
        # print(F.softmax(out), '---', torch.max(F.softmax(out), 1), 'xxx', torch.max(F.softmax(out), 1)[1])
        if step % print_every == print_every-1:
            # softmax ???????????,??max????????:(???,???)
            prediction = torch.max(F.softmax(out, dim=1), 1)[1]
            pred_y = prediction.data.numpy().squeeze()
            target_y = b_y.data.numpy()
            print("pred_y:\t", pred_y)
            print("target_y:\t", target_y)
            logloss = log_loss(target_y, pred_y, eps=1e-15)
            accuracy = sum(pred_y == target_y)/len(target_y)  # ????????????

            # ???
            loop_step = epoch*max_step + step
            total_step = Epoch*max_step
            print('%d-%d %.2f%% (%s) logloss=%.2f \t accuracy=%.2f \t loss=%s' % (epoch, loop_step, loop_step/total_step*100, timeSince(start), logloss, accuracy, loss.data[0]))

            all_losses.append(current_loss/print_every)
            current_loss = 0
     

In [ ]:
from sklearn.metrics import log_loss

Epoch = 1
print_every = 100
max_step = len(X_train_d)

# ???????
current_loss = 0
all_losses = []
pre_result = []
rea_result = []

for epoch in range(Epoch):
    # ??????
    
    # step = len(X_train_d)/BATCH_SIZE (train_loader ? BATCH_SIZE ?????)
    for step, (x, y) in enumerate(zip(X_train_d[1384:], y_train_d[1384:])):
        b_x = prepare_sequence(x)
        b_y = Variable(torch.LongTensor([y]))   # batch y
        
        out = cnn(b_x)                 # ?? net ???? x, ?????
        loss = loss_func(out, b_y)     # ???????

        optimizer.zero_grad()   # ?????????????
        loss.backward()         # ??????, ???????
        optimizer.step()        # ????????? net ? parameters ?

        current_loss += loss.data[0]

        prediction = torch.max(F.softmax(out, dim=1), 1)[1]
        pred_y = prediction.data.numpy()
        target_y = b_y.data.numpy()
        
#         print('??---', pred_y)
#         print('??---', target_y)
            
#         if step>2:
#             break

        # softmax ???????????,??max????????:(???,???)
        prediction = torch.max(F.softmax(out, dim=1), 1)[1]
        pre_result.append(prediction.data.numpy()[0])
        rea_result.append(b_y.data.numpy()[0])
        
        # print(F.softmax(out), '---', torch.max(F.softmax(out), 1), 'xxx', torch.max(F.softmax(out), 1)[1])
        if step % print_every == print_every-1:
#             print('??---', pred_y)
#             print('??---', target_y)
            logloss = log_loss(rea_result, pre_result, eps=1e-15)
#             print("pre_result: \t", pre_result)
#             print("rea_result: \t", rea_result)
            accuracy = sum(np.array(pre_result) == np.array(rea_result))/len(rea_result)  # ????????????
            
            # ???
            loop_step = epoch*max_step + step
            total_step = Epoch*max_step
            
            print('%d-%d %.2f%% (%s) logloss=%.2f \t accuracy=%.2f \t loss=%s' % (epoch, loop_step, loop_step/total_step*100, timeSince(start), logloss, accuracy, loss.data[0]))

            all_losses.append(current_loss/print_every)
            current_loss = 0
            pre_result = []
            rea_result = []


?,train loss?test loss????

* train loss ????,test loss????,????????;
* train loss ????,test loss????,???????;
* train loss ????,test loss????,?????100%???;
* train loss ????,test loss????,????????,????????????;
* train loss ????,test loss????,??????????,?????????,???????????


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.figure()
plt.plot(all_losses)

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.figure()
plt.plot(all_losses)

In [ ]:
# ????????????, ??????,??????(?????)

out_t = cnn(Variable(te_x))

# softmax ???????????,??max????????:(???,???)
prediction_t = torch.max(F.softmax(out_t, dim=1), 1)[1]
pred_t_y = prediction_t.data.numpy().squeeze()
target_t_y = Variable(te_y).data.numpy()
logloss_t = log_loss(target_t_y, pred_t_y, eps=1e-15)
accuracy_t = sum(pred_t_y == target_t_y)/len(target_t_y)  # ????????????
print('(%s) logloss=%.2f \t accuracy=%.2f' % (timeSince(start), logloss_t, accuracy_t))

In [ ]:
# ???????????????,????:???????????,???????????

out_t = cnn(Variable(tr_x[:10000]))

# softmax ???????????,??max????????:(???,???)
prediction_t = torch.max(F.softmax(out_t, dim=1), 1)[1]
pred_t_y = prediction_t.data.numpy().squeeze()
target_t_y = Variable(tr_y[:10000]).data.numpy()
logloss_t = log_loss(target_t_y, pred_t_y, eps=1e-15)
accuracy_t = sum(pred_t_y == target_t_y)/len(target_t_y)  # ????????????
print('(%s) logloss=%.2f \t accuracy=%.2f' % (timeSince(start), logloss_t, accuracy_t))

* * * 

?????

In [ ]:
# ?????????

from gensim.models.word2vec import Word2Vec

model = Word2Vec.load_word2vec_format("vector.txt", binary=False)  # C text format
# model = Word2Vec.load_word2vec_format("vector.bin", binary=True)  # C

In [ ]:
# ?? google ????,????????

from gensim.models.word2vec import Word2Vec 
model = Word2Vec.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

In [ ]:
# ??????

print(model.most_similar(positive=["woman", "king"], negative=["man"], topn=5))
print(model.most_similar(positive=["biggest", "small"], negative=["big"], topn=5))
print(model.most_similar(positive=["ate", "speak"], negative=["eat"], topn=5))

In [ ]:
import numpy as np

with open("food_words.txt", "r") as infile:
    food_words = infile.readlines()
    
with open("sports_words.txt", "r") as infile:
    food_words = infile.readlines()
    
with open("weather_words.txt", "r") as infile:
    food_words = infile.readlines()
    
def getWordVecs(words):
    vec = []
    for word in words:
        word = word.replace("\n", "")
        try:
            vecs.append(model[word].reshape((1, 300)))
        except KeyError:
            continue
    
    # numpy???numpy.concatenate((a1,a2,...), axis=0)????????????????
    """
    >>> a=np.array([1,2,3])
    >>> b=np.array([11,22,33])
    >>> c=np.array([44,55,66])
    >>> np.concatenate((a,b,c),axis=0)  # ?????,axis=0????
    array([ 1,  2,  3, 11, 22, 33, 44, 55, 66]) #????????,axis??????????
    """
    vecs = np.concatenate(vecs)
    return np.array(vecs, dtype="float")

food_vecs = getWordVecs(food_words)
sports_vecs = getWordVecs(sports_words)
weather_vecs = getWordVecs(weather_words)

In [ ]:
# ?? TSNE ? matplotlib ????????????

from sklearn.manifold import TSEN
import matplotlib.pyplot as plt

ts = TSEN(2)
reduced_vecs = ts.fit_transform(np.concatenate((food_vecs, sports_vecs, weather_vecs)))

for i in range(len(reduced_vecs)):
    if i < len(food_vecs):
        color = "b"
    elif i >= len(food_vecs) and i <(len(food_vecs)+len(sports_vecs)):
        color = "r"
    else:
        color = "g"
    
    plt.plot(reduced_vecs[i, 0], reduced_vecs[i, 1], marker="0", color=color, marksize=8)

In [ ]:
# ??,????????? Word2Vec ??:

from sklearn.cross_validation import train_ _test_ _split
from gensim.models.word2vec import Word2Vec

with open('twitter.data/pos_ tweets.txt', 'r') as infile:
    pos_tweets= infile.readlines()

with open(' twitter_ data/neg_ tweets.txt', 'r') as infile:
    neg_ _tweets = infile.readlines()

# use 1for positive sentiment,0 for negative
Y= np.concatenate((np.ones( len (pos_tweets )) ,np.zeros(len(neg_tweets))))

x_train,x_test,y_train,y_test = train_test_split(np.concatenate((pos_tweets, neg_tweets)), y, test_size=0.2)
# Do some very minor text preprocessing

def cleanText(corpus):
    corpus= [z.lower( ).replace(' \n' , '').split() for z in corpus]
    return corpus

x_ train= cleanText(x_ train)
x_ test= cleanText (x_ _test)

n _dim= 300
#Initialize model and build vocab
imdb_w2v= Word2Vec(size=n dim, min_count=10)
imdb_w2v.build_vocab(x_ _train)
#Train the model over train_ _reviews (this may take several minutes)
imdb_w2v.train( x_train)

In [ ]:
# ???,???????????????????????,?????????????????

def buildWordVector(text, size):
    vec = np.zeros(size).reshape((1,size))
    count= 0.

    for word in text :
        try:
            vec += imdb_w2v[word].reshape( (1,size) )
            count += 1.
        except KeyError:
            continue
    if count != 0:
        vec 1'= count
    return vec

In [ ]:
# ????????????????????,??????????????????????,?????????????,????????????????,????????????????????,????????????????????

from sklearn.preprocessing import scale

train_vecs = np.concatenate([buildWordVector(z ,n_dim) for z in x_train])
train_vecs= scale(train_vecs)

# Train word2vec on test tweets
imdb_w2v.train(x_test)


In [ ]:
# ?????????????????????:

#Build test tweet vectors then scale
test_vecs = np.concatenate( [buildWordVector( Z,n _dim) for z in x _test ])
test_vecs = scale(test_vecs)


In [ ]:
"""
???????????????????? ROC ????????????? ROC ??????????????,???????????????????????,???????????????????????,ROC ??????(AUC)??,????????????????????? ROC ?????

(https://en.wikipedia.org/wiki/Receiver_operating_characteristic)

???????????????????????????????
"""

#Use classification algorithm (i.e.Stochastic Logistic Regression) on training set, then assess model performance on test set

from sklearn.linear model import SGDClassifier
lr = SGDClassifier(loss='log' ,penalty='11' )
lr.fit(train_vecs, y_train)
print' Test Accuracy: %.2f' % r.score(test vecs, y_test )




In [ ]:
# ?????? matplotlib ? metric ???? ROC ??

#Crea t e ROC curve
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

pred_probas = lr.predict_proba(test_vecs)[:, 1]

fpr, tpr, _ = roc_curve(y_test, pred_probas )
roc_auc = auc(fpr, tpr)

plt.plot(fpr,tpr,label='area = %.2f' % roc_ auc)
plt.plot([0,1],[0,1],'k--')
plt. xlim( [0. 0 ,1. 0 ])
plt.ylim([0.0, 1.05])
plt.legend(loc='lower right')
plt.show()